# Preprocessing

In [21]:
DATASET = 'Amazon_Sports_and_Outdoors'

REVIEWS_PATH = f'../{DATASET}/reviews_{DATASET[7:]}.json'
META_PATH = f'../{DATASET}/meta_{DATASET[7:]}.json'

INTER_SAVE_PATH = f'../{DATASET}/{DATASET}.inter'
ITEM_SAVE_PATH = f'../{DATASET}/{DATASET}.item'

ITEM_MAPPING_SAVE_PATH = f'../{DATASET}/item_mapping_{DATASET}.json'
USER_MAPPING_SAVE_PATH = f'../{DATASET}/user_mapping_{DATASET}.json'
ITEM_REVERSE_MAPPING_SAVE_PATH = f'../{DATASET}/item_reverse_mapping_{DATASET}.json'
USER_REVERSE_MAPPING_SAVE_PATH = f'../{DATASET}/user_reverse_mapping_{DATASET}.json'

ORGINAL_INTER_PATH = f'../{DATASET}/{DATASET}.inter_orginal'
ORGINAL_ITEM_PATH = f'../{DATASET}/{DATASET}.item_orginal'

DEGENERATE = True
MIN_USER_OCCURRENCE = 5
MIN_ITEM_OCCURRENCE = 5

In [12]:
import pandas as pd
import json
import json5
from tqdm import tqdm

### 1. Generation of .inter file

### 1.1 Reading the 'review' file

In [3]:
data = []
with open(REVIEWS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)

        data.append(
            {
                "user_id:token": record.get("reviewerID"),
                "item_id:token": record.get("asin"),
                "rating:float": record.get("overall"),
                "timestamp:float": record.get("unixReviewTime"),
            }
        )

df = pd.DataFrame(data)
df.head(5)

,user_id:token,item_id:token,rating:float,timestamp:float
0,A3PMSRCL80KSA1,0000031852,4.0,1388275200
1,A1SNLWGLFXD70K,0000031852,4.0,1392940800
2,A1KJ4CVG87QW09,0000031852,4.0,1389657600
3,AA9ITO6ZLZW6,0000031852,5.0,1399507200
4,APJ5ULJ1RMZ4,0000031852,1.0,1398556800


### 1.2 Filter entries with K-Core algorithm

In [4]:
while True:
    last_shape = df.shape

    df = df.groupby('user_id:token').filter(lambda x: len(x) >= MIN_USER_OCCURRENCE)
    df = df.groupby('item_id:token').filter(lambda x: len(x) >= MIN_USER_OCCURRENCE)

    if df.shape == last_shape:
        break

In [5]:
df.shape

(296337, 4)

### 1.3 Generate mappings

In [6]:
df["user_id:token"], user_index = pd.factorize(df["user_id:token"]) # eg. 42 -> "ASFAFASFASF"
df["item_id:token"], item_index = pd.factorize(df["item_id:token"])

reverse_item_index = {org_id: num for num, org_id in enumerate(item_index)} # eg. "ASFAFASFASF" -> 42
reverse_user_index = {org_id: num for num, org_id in enumerate(user_index)}

### 1.4 Validate the dataset

##### Check for null values

In [7]:
df.isna().sum()

user_id:token      0
item_id:token      0
rating:float       0
timestamp:float    0
dtype: int64

### 1.5 Save the .inter file

In [10]:
df.to_csv(INTER_SAVE_PATH, sep="\t", index=False)

### 1.6 Save the mappings to .json file

In [11]:
with open(ITEM_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(list(item_index), fp=f)

with open(ITEM_REVERSE_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(reverse_item_index, fp=f)

with open(USER_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(list(user_index), fp=f)

with open(USER_REVERSE_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(reverse_user_index, fp=f)

### 2. Generation of .item file

### 2.1 Reading the orginal item file and transforming it

In [39]:
df = pd.read_csv(ORGINAL_ITEM_PATH, delimiter='\t')
df = df[df['item_id:token'].isin(reverse_item_index.keys())]
df['item_id:token'] = df['item_id:token'].map(reverse_item_index)
df.head(5)

,item_id:token,title:token,price:float,brand:token,categories:token_seq,sales_type:token,sales_rank:float
132,0,Ghost Inc Glock Armorers Tool 3/32 Punch,9.99,Ghost,"'Sports & Outdoors', 'Hunting', 'Hunting & Fis...",Sports &amp; Outdoors,532941.0
155,1,5 LED Bicycle Rear Tail Red Bike Torch Laser B...,8.26,NaN,"'Lights & Reflectors', 'Sports & Outdoors', 'C...",Toys & Games,15617.0
200,3,Black Mountain Products Resistance Band Set wi...,32.99,Black Mountain,"'Accessories', 'Exercise Bands', 'Sports & Out...",Sports &amp; Outdoors,1010.0
201,2,Black Mountain Products Single Resistance Band...,10.49,Black Mountain,"'Accessories', 'Exercise Bands', 'Sports & Out...",Sports &amp; Outdoors,1010.0
282,4,Outers Universal 32-Piece Blow Molded Gun Clea...,21.99,Outers,"'Sports & Outdoors', 'Hunting', 'Gun Cleaning ...",Sports &amp; Outdoors,26457.0


### 2.2 Validate the dataset

##### Check for null values

In [40]:
df.isna().sum()

item_id:token              0
title:token               91
price:float             3593
brand:token             6078
categories:token_seq       0
sales_type:token        1878
sales_rank:float        1878
dtype: int64

### 2.3 Save the .item file

In [41]:
df.to_csv(ITEM_SAVE_PATH, sep='\t', index=False, na_rep='')